---
categories: [MongoDB]
---

# MongoDB Exercises

<img src="nosql_cartoon.jpeg" width="500">

These are some MongoDB exercises I did for DSCI 513 at UBC.

In [ ]:
from pymongo import MongoClient
import json
import urllib.parse

with open('data/credentials_mongodb.json') as f:
    login = json.load(f)

username = login['username']
password = urllib.parse.quote(login['password'])
host = login['host']
url = "mongodb+srv://{}:{}@{}/?retryWrites=true&w=majority".format(username, password, host)
client = MongoClient(url)

## Exercise 1: Getting to know your MongoDB databases
---

### 1.1

rubric={accuracy:1}

List the databases that exist on your MongoDB Atlas cluster after loading sample databases. You can do this either by checking out the databases using Compass or by using `pymongo`'s `.list_database_names()` method.

In [ ]:
client.list_database_names()

['sample_airbnb',
 'sample_analytics',
 'sample_geospatial',
 'sample_guides',
 'sample_mflix',
 'sample_restaurants',
 'sample_supplies',
 'sample_training',
 'sample_weatherdata',
 'admin',
 'local']

### 1.2

rubric={accuracy:1}

List the collections stored in the `sample_mflix` and `sample_airbnb` databases. You can do this either by checking out the databases using Compass or by using `pymongo`'s `.list_collection_names()` method.

In [ ]:
print(client["sample_mflix"].list_collection_names(), "\n",
client["sample_airbnb"].list_collection_names(), sep="")

['users', 'sessions', 'comments', 'embedded_movies', 'movies', 'theaters']
['listingsAndReviews']


### 1.3

rubric={accuracy:2}

We would like to create a summary report to get the number of documents in each collection of each database on our cluster. Write a python loop that produces an output similar to the following:

```
Database: sample_airbnb
(collection, n_docs) =  ('listingsAndReviews', 5555)

Database: sample_analytics
(collection, n_docs) =  ('transactions', 1746)
(collection, n_docs) =  ('accounts', 1746)
(collection, n_docs) =  ('customers', 500)
.
.
.
```

In previous questions, you've been introduced to the two methods of `pymongo` which are used to return database and collection names. Moreover, you can use the `.count_documents(filter={}))` method to count all documents in each collection.

In [ ]:
for dbs in client.list_database_names():
    print(f"Database: {dbs}")
    for coll in client[dbs].list_collection_names():
        print(f"(collection, n_docs) = ({coll}, {client[dbs][coll].count_documents(filter={})})")
    print("")

Database: sample_airbnb
(collection, n_docs) = (listingsAndReviews, 5555)

Database: sample_analytics
(collection, n_docs) = (customers, 500)
(collection, n_docs) = (accounts, 1746)
(collection, n_docs) = (transactions, 1746)

Database: sample_geospatial
(collection, n_docs) = (shipwrecks, 11095)

Database: sample_guides
(collection, n_docs) = (planets, 8)

Database: sample_mflix
(collection, n_docs) = (users, 185)
(collection, n_docs) = (sessions, 1)
(collection, n_docs) = (comments, 41079)
(collection, n_docs) = (embedded_movies, 3483)
(collection, n_docs) = (movies, 21349)
(collection, n_docs) = (theaters, 1564)

Database: sample_restaurants
(collection, n_docs) = (neighborhoods, 195)
(collection, n_docs) = (restaurants, 25359)

Database: sample_supplies
(collection, n_docs) = (sales, 5000)

Database: sample_training
(collection, n_docs) = (grades, 100000)
(collection, n_docs) = (zips, 29470)
(collection, n_docs) = (companies, 9500)
(collection, n_docs) = (inspections, 80047)
(colle

## Exercise 2: Basic MongoDB queries
---

### 2.1

rubric={accuracy:1}

Retrieve one (random) document associated with a movie produced in 2015.

You can use `.find_one()` method to do this, or use `.find()` but limit your results to 1 document.

In [ ]:
client["sample_mflix"]["movies"].find_one(filter = {"year": 2015})

{'_id': ObjectId('573a13adf29313caabd2b765'),
 'plot': "A new theme park is built on the original site of Jurassic Park. Everything is going well until the park's newest attraction--a genetically modified giant stealth killing machine--escapes containment and goes on a killing spree.",
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'runtime': 124,
 'metacritic': 59,
 'rated': 'PG-13',
 'cast': ['Chris Pratt',
  'Bryce Dallas Howard',
  'Irrfan Khan',
  "Vincent D'Onofrio"],
 'num_mflix_comments': 0,
 'poster': 'https://m.media-amazon.com/images/M/MV5BNzQ3OTY4NjAtNzM5OS00N2ZhLWJlOWUtYzYwZjNmOWRiMzcyXkEyXkFqcGdeQXVyMTMxODk2OTU@._V1_SY1000_SX677_AL_.jpg',
 'title': 'Jurassic World',
 'fullplot': '22 years after the original Jurassic Park failed, the new park (also known as Jurassic World) is open for business. After years of studying genetics the scientists on the park genetically engineer a new breed of dinosaur. When everything goes horribly wrong, will our heroes make it off the island

### 2.2

rubric={accuracy:1}

Retrieve all TV series produced in 1995.

> **Hint:** Inspect possible values for the `type` field.

In [ ]:
list(client["sample_mflix"]["movies"].find(filter = {"year": 1995, "type":"TV"}))

[]

### 2.3

rubric={accuracy:2}

Retrieve the title and cast of movies produced in 2010, but limit your results to 5 documents.

> **Note:** Don't forget to exclude the `_id` field from your returned documents.

> **Note:** The exact returned documents returned by `pymongo` might be different in different sessions and on different computers. This is perfectly fine.

In [ ]:
list(client["sample_mflix"]["movies"].find(
    filter = {"year": 2010},
    projection = {'_id':0, 'title':1, 'cast':1},
    limit = 5
    )
)

[{'cast': ['èva Gèbor', 'Istvèn Znamenèk', 'èkos Horvèth', 'Lia Pokorny'],
  'title': 'Pèl Adrienn'},
 {'title': 'In My Sleep',
  'cast': ['Philip Winchester',
   'Tim Draxl',
   'Lacey Chabert',
   'Abigail Spencer']},
 {'cast': ['James Badge Dale',
   'Joseph Mazzello',
   'Jon Seda',
   'Sebastian Bertoli'],
  'title': 'The Pacific'},
 {'cast': ['Mandy Moore', 'Zachary Levi', 'Donna Murphy', 'Ron Perlman'],
  'title': 'Tangled'},
 {'cast': ['Nikita Mikhalkov',
   'Oleg Menshikov',
   'Nadezhda Mikhalkova',
   'Sergey Makovetskiy'],
  'title': 'Utomlyonnye solntsem 2: Predstoyanie'}]

### 2.4

rubric={accuracy:2}

Retrieve the top 15 movies produced in 2010 that have the longest duration. Exclude TV series from your results. The returned documents should only contain the `title` and `runtime` fields (exclude the `_id` field).

> **Note:** It's ok if your results contain duplicate movies.

In [ ]:
list(client["sample_mflix"]["movies"].find(
    filter = {'year': 2010, 'type':{"$ne":"TV"}},
    projection = {'_id':0, 'title':1, 'runtime':1, 'metacritic':1},
    sort=[('runtime', -1), ('metacritic', -1)]
    )
)

[{'runtime': 272, 'metacritic': 82, 'title': 'Mysteries of Lisbon'},
 {'runtime': 185, 'title': 'Going Postal'},
 {'runtime': 181, 'metacritic': 63, 'title': 'Aurora'},
 {'runtime': 181, 'title': 'Utomlyonnye solntsem 2: Predstoyanie'},
 {'runtime': 180,
  'metacritic': 87,
  'title': 'The Autobiography of Nicolae Ceausescu'},
 {'runtime': 180, 'title': 'Thorne: Sleepyhead'},
 {'runtime': 178, 'title': 'Riverworld'},
 {'runtime': 174, 'title': 'Enthiran'},
 {'runtime': 170, 'title': 'We Believed'},
 {'runtime': 170, 'title': 'Khaleja'},
 {'metacritic': 50, 'title': 'My Name Is Khan', 'runtime': 165},
 {'runtime': 163, 'title': 'Moss'},
 {'runtime': 163, 'title': 'Raajneeti'},
 {'runtime': 160, 'title': 'Singam'},
 {'runtime': 159, 'title': 'Black Venus'},
 {'runtime': 157, 'title': 'Will You Cross the Skies for Me?'},
 {'runtime': 157, 'title': 'Will You Cross the Skies for Me?'},
 {'runtime': 152, 'title': 'Sketches of Kaitan City'},
 {'runtime': 151, 'title': 'Anjaana Anjaani'},
 {'r

### 2.5

rubric={accuracy:2}

For year 2015, return the number of movies with a metacritic rating of exactly 90.

In [ ]:
movies = client["sample_mflix"]["movies"]
movies.count_documents(
    filter={'metacritic':90}
)

40

### 2.6

rubric={accuracy:2}

Retrieve the title and runtime of the 10 shortest movies in the `movies` collection.

For this exercise, you need to make sure that the field `runtime` exists in the returned documents, otherwise by default those documents would appear first which don't have a `runtime` field at all!

> **Hint:** You need the `$exists` operator (see [here](https://docs.mongodb.com/manual/reference/operator/query/exists/) for help).

In [ ]:
list(movies.find(
    filter = {'runtime' :{'$exists':1}},
    sort = [('runtime', 1)],
    projection = {'_id':0, 'title':1, 'runtime':1},
    limit = 10
))

[{'runtime': 1, 'title': 'Neko no shukai'},
 {'runtime': 1, 'title': 'The Kiss'},
 {'runtime': 1, 'title': 'The Kiss'},
 {'runtime': 2, 'title': 'Fresh Guacamole'},
 {'runtime': 2, 'title': 'Pixels'},
 {'runtime': 2, 'title': 'Game Over'},
 {'runtime': 2, 'title': 'Andrè and Wally B.'},
 {'runtime': 2, 'title': 'Luxo Jr.'},
 {'runtime': 3, 'title': 'Sisyphus'},
 {'runtime': 3, 'title': 'Gagarin'}]

## Exercise 3: Conditionals, embedded documents & arrays
---

### 3.1

rubric={accuracy:2}

Retrieve the title, production year, and number of awards of all movies that

- have been produced between 1950 and 2000 (inclusive)
- have an IMDB rating of 8.5 or better
- won at least 30 awards.

Sort the results by production year in descending order.

In [ ]:
list(
    movies.find(
        filter = {'year': {"$gte": 1905, "$lte": 2000},
                  'awards.wins': {'$gte': 30},
                  'imdb.rating': {'$gte': 8.5},
                  },
        projection = {'_id':0, 'title':1, 'year':1, 'awards.wins': 1},
        sort = {'year':-1}
    )
)

[{'year': 2000, 'title': 'Memento', 'awards': {'wins': 54}},
 {'year': 2000, 'title': 'Gladiator', 'awards': {'wins': 63}},
 {'year': 1999, 'title': 'The Matrix', 'awards': {'wins': 37}},
 {'year': 1998, 'title': 'Saving Private Ryan', 'awards': {'wins': 83}},
 {'title': 'Life Is Beautiful', 'awards': {'wins': 66}, 'year': 1997},
 {'year': 1997, 'title': 'Life Is Beautiful', 'awards': {'wins': 66}},
 {'year': 1995, 'title': 'The Usual Suspects', 'awards': {'wins': 36}},
 {'year': 1995, 'title': 'Se7en', 'awards': {'wins': 32}},
 {'year': 1994, 'title': 'Pulp Fiction', 'awards': {'wins': 64}},
 {'year': 1994, 'title': 'Forrest Gump', 'awards': {'wins': 46}},
 {'title': "Schindler's List", 'awards': {'wins': 81}, 'year': 1993},
 {'year': 1991, 'title': 'The Silence of the Lambs', 'awards': {'wins': 56}},
 {'year': 1990, 'title': 'Goodfellas', 'awards': {'wins': 43}},
 {'year': 1981, 'title': 'Raiders of the Lost Ark', 'awards': {'wins': 32}},
 {'year': 1977,
  'title': 'Star Wars: Episod

### 3.2

rubric={accuracy:2}

Find the top 15 highest-rated movies according to IMDB for movies that have at least 100,000 votes. Your returned documents should only contain the `title`, `year`, and `imdb.rating` fields.

> **Hint:** Be careful about documents which have a blank space in their `imdb.rating` field!

> **Note:** It's ok if your results contain duplicates. Return 15 documents in any case.

In [ ]:
list(
    movies.find(
        filter = {
            "imdb.votes": {"$gte": 100000, "$exists": 1}
        },
        projection = {"_id":0, "title":1, "year":1, "imdb.rating":1},
        sort = [("imdb.rating", -1)],
        limit=15
    )
)

[{'title': 'Band of Brothers', 'year': 2001, 'imdb': {'rating': 9.6}},
 {'imdb': {'rating': 9.3}, 'year': 1994, 'title': 'The Shawshank Redemption'},
 {'imdb': {'rating': 9.3}, 'year': 1994, 'title': 'The Shawshank Redemption'},
 {'imdb': {'rating': 9.2}, 'year': 1972, 'title': 'The Godfather'},
 {'imdb': {'rating': 9.1}, 'year': 1974, 'title': 'The Godfather: Part II'},
 {'imdb': {'rating': 9.0}, 'year': 2008, 'title': 'The Dark Knight'},
 {'imdb': {'rating': 8.9}, 'year': 1999, 'title': 'Fight Club'},
 {'imdb': {'rating': 8.9}, 'year': 1994, 'title': 'Pulp Fiction'},
 {'imdb': {'rating': 8.9},
  'year': 2003,
  'title': 'The Lord of the Rings: The Return of the King'},
 {'title': "Schindler's List", 'year': 1993, 'imdb': {'rating': 8.9}},
 {'imdb': {'rating': 8.8},
  'year': 1980,
  'title': 'Star Wars: Episode V - The Empire Strikes Back'},
 {'imdb': {'rating': 8.8},
  'year': 2001,
  'title': 'The Lord of the Rings: The Fellowship of the Ring'},
 {'imdb': {'rating': 8.8}, 'year': 2

### 3.3

rubric={accuracy:1}

Retrieve the title, production year, and IMDB rating of movies in which both **Morgan Freeman** and **Clint Eastwood** played a role (among other actors in those movies). Sort the returned documents by year in descending order.

In [ ]:
list(
    movies.find(
        filter = {"cast":{"$all":["Morgan Freeman", "Clint Eastwood"]}},
        projection = {'_id':0, 'title':1, 'year':1, 'imdb.rating':1, "cast":1},
        sort = {"year":-1}
    )
)

[{'imdb': {'rating': 8.1},
  'year': 2004,
  'title': 'Million Dollar Baby',
  'cast': ['Clint Eastwood',
   'Hilary Swank',
   'Morgan Freeman',
   'Jay Baruchel']},
 {'imdb': {'rating': 8.3},
  'year': 1992,
  'title': 'Unforgiven',
  'cast': ['Clint Eastwood',
   'Gene Hackman',
   'Morgan Freeman',
   'Richard Harris']}]

### 3.4

rubric={accuracy:3}

Retrieve documents associated with movies which:

- are available in both German and French (among other languages), but not in English,
- are either rated above 8 according to IMDB, or above 7.5 according to the critic ratings of [Rotten Tomatoes](https://www.rottentomatoes.com/) (inspect the `tomatoes` field),
- have at least 50 Rotten Tomatoes critic reviews.

The returned documents should include the title, year, IMDB rating, Rotten Tomatoes critic rating and country of production fields. Sort the results by IMDB rating in descending order.

**Note:** Duplicates in the results are ok.

In [ ]:
list(movies.find(
    filter = {"languages": {"$all":["German", "French"], "$nin": ["English"]},
              "$or": [
                  {'imdb.rating': {"$gt": 8}},
                  {'tomatoes.critic.rating': {"$gt": 7.5}}],
                "tomatoes.critic.numReviews": {"$gte": 50}
              },
    projection = {"_id":0, "title":1, "year":1, "imdb.rating":1, "tomatoes.critic.rating": 1, "countries":1},
    sort = [("imdb.rating", -1)]
))

[{'imdb': {'rating': 7.8},
  'year': 2013,
  'title': 'The Wind Rises',
  'tomatoes': {'critic': {'rating': 7.9}},
  'countries': ['Japan']},
 {'title': 'The Wind Rises',
  'year': 2013,
  'imdb': {'rating': 7.8},
  'countries': ['Japan'],
  'tomatoes': {'critic': {'rating': 7.9}}}]

### 3.5

rubric={accuracy:2}

Find the title and production year of the top 20 award-winning movies which have **not** been produced in USA, Canada, UK, or Australia.

**Note:** Duplicates are ok. Return 20 documents in any case.

In [ ]:
list(movies.find(
    filter = {"countries": {"$nin": ["USA", "Canada", "UK", "Australia"]}},
    limit=20,
    projection = {"_id":0, "title":1, "year":1},
    sort=[("awards.wins", -1)]
))

[{'title': 'The Artist', 'year': 2011},
 {'title': 'Amour', 'year': 2012},
 {'title': 'Amour', 'year': 2012},
 {'title': 'A Separation', 'year': 2011},
 {'year': 2006, 'title': 'The Lives of Others'},
 {'title': 'Let the Right One In', 'year': 2008},
 {'year': 2002, 'title': 'City of God'},
 {'year': 2006, 'title': 'Volver'},
 {'title': 'Life Is Beautiful', 'year': 1997},
 {'title': 'The Sea Inside', 'year': 2004},
 {'year': 1997, 'title': 'Life Is Beautiful'},
 {'year': 2009, 'title': 'The White Ribbon'},
 {'year': 2001, 'title': 'Amèlie'},
 {'year': 2000, 'title': 'Amores Perros'},
 {'title': 'Shall We Dance?', 'year': 1996},
 {'title': 'Shall We Dance?', 'year': 1996},
 {'year': 2013, 'title': 'The Grandmaster'},
 {'year': 1999, 'title': 'All About My Mother'},
 {'year': 2001, 'title': 'Spirited Away'},
 {'year': 2009, 'title': 'The Secret in Their Eyes'}]